<a href="https://colab.research.google.com/github/LivingstonTardzenyuy/Generative-AI/blob/main/basic_rag_implimentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importing our open AI keys.

In [1]:
from google.colab import userdata
grok_key = userdata.get('GROK_api')

open_ai = userdata.get('OPENAI_API_KEY')

## Performing our Chunks - Embedding model

In [5]:
!pip install langchain-community
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 4.1 MB/s eta 0:00:00


In [6]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter

from langchain_community.document_loaders import TextLoader # to load our custom data

In [8]:
import os
import gdown

# The folder 'data' should already exist from previous steps
folder_name = 'data'
os.makedirs(folder_name, exist_ok=True)

# List of files to download
files_to_download = [
    {
        'url': 'https://docs.google.com/document/d/1GgsuYbVOMhFX485g9DfQCCga5F0wFupC2xye-C7J5QQ/edit?tab=t.0',
        'type': 'doc',
        'output_name': 'document1.txt'
    },
    {
        'url': 'https://docs.google.com/spreadsheets/d/1a-UcUgnd2YFqb8pj0bKMxN9FEk9qNO4qJKYTnoWjsdA/edit?gid=0#gid=0',
        'type': 'sheet',
        'output_name': 'spreadsheet1.csv'
    },
    {
        'url': 'https://docs.google.com/document/d/1EeLapuw8QVuVFSAkeCETGKOPi0yUT7PaJ0wgaF8hD7A/edit?tab=t.0#heading=h.mbra3ggubsrw',
        'type': 'doc',
        'output_name': 'document2.txt'
    }
]

for file_info in files_to_download:
    url = file_info['url']
    file_type = file_info['type']
    output_name = file_info['output_name']

    # Extract document/spreadsheet ID
    if 'document/d/' in url:
        document_id = url.split('document/d/')[1].split('/')[0]
    elif 'spreadsheets/d/' in url:
        document_id = url.split('spreadsheets/d/')[1].split('/')[0]
    else:
        print(f"Skipping {url}: Unrecognized Google Drive URL format.")
        continue

    # Construct export URL based on file type
    if file_type == 'doc':
        export_url = f'https://docs.google.com/document/d/{document_id}/export?format=txt'
    elif file_type == 'sheet':
        export_url = f'https://docs.google.com/spreadsheets/d/{document_id}/export?format=csv'
    else:
        print(f"Skipping {url}: Unsupported file type {file_type}.")
        continue

    # Define the full path for the downloaded file
    output_file_path = os.path.join(folder_name, output_name)

    print(f"\nDownloading {output_name} from {url}...")
    gdown.download(export_url, output_file_path, quiet=False)
    print(f"File downloaded to: {output_file_path}")

Downloading...
From: https://docs.google.com/document/d/1GgsuYbVOMhFX485g9DfQCCga5F0wFupC2xye-C7J5QQ/export?format=txt
To: /content/data/document1.txt
7.48kB [00:00, 17.0MB/s]


File downloaded to: data/document1.txt



Downloading...
From: https://docs.google.com/spreadsheets/d/1a-UcUgnd2YFqb8pj0bKMxN9FEk9qNO4qJKYTnoWjsdA/export?format=csv
To: /content/data/spreadsheet1.csv
8.96kB [00:00, 2.64MB/s]


File downloaded to: data/spreadsheet1.csv



Downloading...
From: https://docs.google.com/document/d/1EeLapuw8QVuVFSAkeCETGKOPi0yUT7PaJ0wgaF8hD7A/export?format=txt
To: /content/data/document2.txt
2.78kB [00:00, 7.17MB/s]

File downloaded to: data/document2.txt


## Creating our chunks. Splitting the data into smaller components

In [9]:
text_splitter = CharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 0
)

# Define the paths to your files
file_paths = [
    'data/document1.txt',
    'data/spreadsheet1.csv',
    'data/document2.txt'
]

# Load each document using TextLoader
# TextLoader can handle various file types, treating them as plain text
documents = []
for file_path in file_paths:
    loader = TextLoader(file_path)
    documents.extend(loader.load())

chunks_of_text = text_splitter.split_documents(documents)

# Display the first few chunks to verify
for i, chunk in enumerate(chunks_of_text[:3]):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content)
    print("\n")

--- Chunk 1 ---
﻿CAITCC Vocational Training Program


Module: Introduction to Prompt Engineering


Prepared by: Kongnyuy Livingston & Nyuydini Bill


For: CAITCC Vocational Training Center


Programs: AI Product Manager | AI Product Marketing Manager | AI Engineer
Phase 1: Understanding the Basics of Programming (Using Python)
(For CAITCC Vocational Training – AI Product Management Program)
________________


1. Introduction to Programming
Programming means giving a computer a set of instructions to perform a specific task.
It’s like teaching a computer to solve a problem step by step.
Just as we use English or French to talk to people, programmers use languages like Python to communicate with computers.
💡 Why Python?
Python is a beginner-friendly programming language that is:
* Simple and readable — it looks almost like English.

* Used everywhere — in AI, data science, web development, and automation.

* Great for building problem-solving and logical thinking skills.


--- Chunk 2 --

In [10]:
len(chunks_of_text)

15

## Loading our Embeddding model

we will create an embedding model, then later store it in a FAISS database

In [16]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 100.3 MB/s eta 0:00:00


In [19]:
embeddings = OpenAIEmbeddings(
    openai_api_key = open_ai,
    # model_name = 'text-embedding-ada-002'
)

vector_db = FAISS.from_documents(chunks_of_text, embeddings)

In [21]:
# getting the retriever. and by default it will return 4 parameters.
retriever = vector_db.as_retriever()

response = retriever.invoke("What is CAITTC ? ")
print(response)

[Document(id='b0459a5a-249d-436c-b269-5976ec990711', metadata={'source': 'data/document1.txt'}, page_content='\ufeffCAITCC Vocational Training Program\n\n\nModule: Introduction to Prompt Engineering\n\n\nPrepared by: Kongnyuy Livingston & Nyuydini Bill\n\n\nFor: CAITCC Vocational Training Center\n\n\nPrograms: AI Product Manager | AI Product Marketing Manager | AI Engineer\nPhase 1: Understanding the Basics of Programming (Using Python)\n(For CAITCC Vocational Training – AI Product Management Program)\n________________\n\n\n1. Introduction to Programming\nProgramming means giving a computer a set of instructions to perform a specific task.\nIt’s like teaching a computer to solve a problem step by step.\nJust as we use English or French to talk to people, programmers use languages like Python to communicate with computers.\n💡 Why Python?\nPython is a beginner-friendly programming language that is:\n* Simple and readable — it looks almost like English.\n\n* Used everywhere — in AI, data 

In [22]:
len(response)

4

## Simple use with Langchain Expression Language(LCEL)

In [23]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough



In [24]:
# Creating our prompt template.
template = """
            Answer the question based only on the following context:
            {context}

            Question: {question}
          """

prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI(
    openai_api_key = open_ai,
)

In [27]:
# format the output.
def format_docs(docs):
  return "\n\n".join([d.page_content for d in docs])


chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)


In [29]:
response = chain.invoke(
    "what is the purpose of CAITTC ?"
)
print(response)

The purpose of CAITTC is to provide vocational training in AI Product Management, AI Product Marketing Management, and AI Engineering, with a focus on understanding the basics of programming using Python.


In [32]:
response = chain.invoke(
    "what is CAITCC. Do you know their teachers  ?"
)
print(response)

CAITCC stands for CAITCC Vocational Training Center, and it offers programs such as AI Product Manager, AI Product Marketing Manager, and AI Engineer. The teachers mentioned in the context are Kongnyuy Livingston and Nyuydini Bill.
